# 04 — Positional Encoding
### Starter Notebook

**Learning objectives**
- Understand why attention is permutation-invariant and needs positional information
- Implement sinusoidal positional encoding
- Compare sinusoidal vs learned vs RoPE
- Visualise positional patterns

---

In [ ]:
import sys, os, math
sys.path.insert(0, os.path.abspath('../..'))
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional

torch.manual_seed(7)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Part A — The Permutation Problem

Attention treats its input as a **set**, not a sequence. The output is the same regardless of token order — unless we encode position explicitly.

### Exercise A1 — Demonstrate permutation invariance

In [ ]:
from src.models.attention import MultiHeadAttention

mha = MultiHeadAttention(d_model=32, n_heads=2)
mha.eval()

x  = torch.randn(1, 6, 32)           # original sequence
perm = [3, 0, 5, 1, 4, 2]
x_perm = x[:, perm, :]               # shuffled tokens

with torch.no_grad():
    out,  _ = mha(x,      x,      x)
    out_p, _ = mha(x_perm, x_perm, x_perm)

# TODO: are the outputs permuted versions of each other?
# Compare out[:, perm, :] with out_p and print whether they match.
print('[Exercise A1] Show that attention output is the same set, just reordered.')

## Part B — Sinusoidal Positional Encoding

From the original Transformer paper. Each position $p$ and each dimension $i$ gets:

$$PE(p, 2i)   = \sin\!\left(\frac{p}{10000^{2i/d}}\right)$$
$$PE(p, 2i+1) = \cos\!\left(\frac{p}{10000^{2i/d}}\right)$$

### Exercise B1 — Implement from scratch

In [ ]:
def sinusoidal_encoding(seq_len: int, d_model: int) -> torch.Tensor:
    """
    Returns positional encoding matrix of shape (seq_len, d_model).

    Steps:
    1. Create position vector [0, 1, ..., seq_len-1]
    2. Create dimension indices [0, 2, 4, ...] for sin, [1, 3, 5, ...] for cos
    3. Compute div_term = 10000^(2i/d_model)
    4. Fill even cols with sin, odd cols with cos
    """
    pe = torch.zeros(seq_len, d_model)

    # TODO: implement the formula above

    return pe


pe = sinusoidal_encoding(seq_len=50, d_model=64)
print(f'PE shape: {pe.shape}   # expect (50, 64)')

plt.figure(figsize=(12, 4))
plt.imshow(pe.numpy().T, cmap='RdBu', aspect='auto', vmin=-1, vmax=1)
plt.xlabel('Position')
plt.ylabel('Dimension')
plt.title('Sinusoidal Positional Encoding')
plt.colorbar()
plt.tight_layout()
plt.show()

### Exercise B2 — Relative position via dot products

One nice property: the dot product `PE(p) · PE(q)` is a function of `|p-q|`, not absolute positions. Verify this.

In [ ]:
if pe.sum() != 0:   # skip if not implemented
    # Compute the full (seq, seq) similarity matrix of PE vectors
    # TODO: sim = pe @ pe.T
    sim = None

    if sim is not None:
        plt.figure(figsize=(6, 5))
        plt.imshow(sim.numpy(), cmap='RdBu', aspect='auto')
        plt.xlabel('Position j'); plt.ylabel('Position i')
        plt.title('PE dot-product similarity — only depends on |i-j|')
        plt.colorbar()
        plt.tight_layout()
        plt.show()

## Part C — Compare Encodings

Use the library implementations to compare sinusoidal, learned, and RoPE.

In [ ]:
from src.models.embeddings import (
    SinusoidalPositionalEncoding,
    LearnedPositionalEmbedding,
    RotaryPositionalEmbedding,
)

d_model, seq_len = 64, 100

sin_pe = SinusoidalPositionalEncoding(d_model, seq_len)
learned_pe = LearnedPositionalEmbedding(seq_len, d_model)

sin_enc = sin_pe.get_encoding(seq_len).detach().numpy()       # (seq, d_model)
learned_enc = learned_pe.get_encoding(seq_len).detach().numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, enc, title in zip(
    axes,
    [sin_enc, learned_enc],
    ['Sinusoidal (fixed)', 'Learned (random init)']
):
    im = ax.imshow(enc.T, cmap='RdBu', aspect='auto', vmin=-1, vmax=1)
    ax.set_xlabel('Position'); ax.set_ylabel('Dimension')
    ax.set_title(title)
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

print('Key difference: sinusoidal generalises to longer sequences at test time;')
print('learned embeddings are capped at max_seq_len but may fit training data better.')

## Part D — Rotary Position Embedding (RoPE)

RoPE (used in LLaMA, Gemma) encodes position *inside* the attention computation by rotating Q and K vectors. This gives relative position encoding with better length generalisation.

In [ ]:
rope = RotaryPositionalEmbedding(d_model=64, max_seq_len=256)

# Dummy Q, K: (batch=1, heads=1, seq=20, d_k=64)
q = torch.randn(1, 1, 20, 64)
k = torch.randn(1, 1, 20, 64)

q_rot, k_rot = rope(q, k)
print(f'Q shape before/after RoPE: {q.shape} → {q_rot.shape}')

# Verify that the rotation preserves vector norms
q_norms     = q.norm(dim=-1)
q_rot_norms = q_rot.norm(dim=-1)
print(f'Norm preserved: {torch.allclose(q_norms, q_rot_norms, atol=1e-5)}')

# TODO: visualise how q_rot[0, 0] differs from q[0, 0] — plot the first two dims
print('[Optional] Plot the rotation effect on the first two dimensions across positions.')

## Summary

| Encoding | Fixed? | Length generalisation | Used by |
|----------|--------|-----------------------|---------|
| Sinusoidal | Yes | Good (extrapolates) | Original Transformer |
| Learned | No | Limited to max_len | GPT-2, BERT |
| RoPE | Yes | Excellent | LLaMA, Gemma, Mistral |

**Next:** `05_feedforward_network_starter.ipynb` — the other half of the transformer block.